# Getting started

This guide is a quick tour of the package: what a motif looks like here, how
to convert between its different forms, how to read and write motif files,
and how to use motifs to scan and compare. Each section only scratches the
surface, and the [API reference](../reference/index.md) has the details.

Everything is imported from the top-level package:

In [ ]:
import universalmotif as um

## Motifs

A motif can be made from a consensus string, from a set of aligned sites, or
from a matrix. For example, here is one made from three sites:

In [ ]:
sites = ["TGACTCA", "TGAGTCA", "TGACTAA"]
ap1 = um.create_motif(sites, pseudocount=1, name="AP-1")
print(ap1)

Printing a motif shows its metadata and its matrix, with one column per
position and the consensus letters as column names. Note that the
consensus uses the IUPAC ambiguity codes, so the S at position 4 means C or
G. The metadata and matrix are all available as attributes:

In [ ]:
print(ap1.name, ap1.alphabet, ap1.type, ap1.consensus)
print(round(ap1.icscore, 2))
ap1.matrix.round(2)

In a Jupyter notebook, a motif on its own at the end of a cell is shown
as a logo next to its metadata, with the matrix tucked away underneath.
[`logo_svg()`][universalmotif.logo_svg] gives the same logo as an SVG
image for reports and web pages:

In [ ]:
ap1

Motifs are immutable, so functions always return new motifs rather than
changing the ones they are given. To change some of the metadata, use
`replace()`:

In [ ]:
ap1 = ap1.replace(family="bZIP", organism="Homo sapiens")
print(ap1.family, ap1.organism)

## Motif types

The same motif can be represented as counts, probabilities, log-odds
weights or information content, which are the PCM, PPM, PWM and ICM types
respectively. Converting between them
is a single call to [`convert_type()`][universalmotif.convert_type]. Converting to counts uses the number
of sites the motif was made from, and converting to weights uses its
pseudocount to keep zero probabilities from becoming minus infinity:

In [ ]:
pwm = um.convert_type(ap1, "PWM")
pwm.matrix.round(2)

In [ ]:
um.convert_type(ap1, "ICM").matrix.round(2)

## Reading and writing motifs

There are readers and writers for the MEME, JASPAR, HOMER, TRANSFAC, CIS-BP,
UniPROBE and plain matrix formats. Let us write two motifs to a MEME file and
read them back. Readers always return a list, even for a single motif:

In [ ]:
import tempfile, pathlib

creb_sites = ["TGACGTCA", "TGACGTCA", "TTACGTAA"]
creb = um.create_motif(creb_sites, pseudocount=1, name="CREB")
path = pathlib.Path(tempfile.mkdtemp()) / "motifs.meme"
um.write_meme([ap1, creb], path)
motifs = um.read_meme(path)
[m.name for m in motifs]

[`summarise_motifs()`][universalmotif.summarise_motifs] gives an overview of a list of motifs as a
DataFrame, which is handy once there are more than a few of them:

In [ ]:
um.summarise_motifs(motifs)

## Changing motifs

A few of the functions for working with motifs. [`motif_rc()`][universalmotif.motif_rc] reverse
complements a motif, [`trim_motifs()`][universalmotif.trim_motifs] removes uninformative positions from
its edges, and [`merge_motifs()`][universalmotif.merge_motifs] aligns several motifs and combines them into
one:

In [ ]:
print(um.motif_rc(ap1).consensus)
loose = um.create_motif("NNTGACTCANN", pseudocount=1)
print(um.trim_motifs(loose).consensus)
longer = um.create_motif(["ATGAGTCAT"], pseudocount=1)
print(um.merge_motifs([ap1, longer], min_overlap=5).consensus)

## Logos

[`view_motifs()`][universalmotif.view_motifs] draws sequence logos with matplotlib, aligning several
motifs to one another first and reverse complementing them where that
lines them up better:

In [ ]:
fig = um.view_motifs([ap1, creb])

## Scanning sequences

Let us make some random sequences and plant an AP-1 site in a few of them,
then scan for both motifs. [`scan_sequences()`][universalmotif.scan_sequences] returns one row per hit, each
with its exact P-value; only hits at or below the `pvalue` cutoff are
kept:

In [ ]:
import polars as pl

seqs = um.create_sequences(seqnum=5, seqlen=200, seed=1)
for i in (0, 2, 3):
    seqs[i] = seqs[i][:100] + "TGACTCA" + seqs[i][107:]
hits = um.scan_sequences([ap1, creb], seqs, pvalue=1e-3)
with pl.Config(tbl_rows=20):
    display(hits.select("motif", "sequence", "start", "strand", "match", "pvalue"))

Each planted site is found on both strands, since TGACTCA
reads as TGAGTCA on the reverse strand, which matches the motif almost as
well. The other hits are chance matches in the random sequence: one
AP-1-like site in seq1 and two CREB-like ones in seq3, each of them close
enough to palindromic to turn up on both strands too. A handful of these
is to be expected at a cutoff as lenient as 1e-3, which is why stricter
cutoffs such as 1e-4 or 1e-5 are more usual for real sequences.

Sequences can also be read from a FASTA file with [`read_fasta()`][universalmotif.read_fasta], and very
large ones, such as a whole genome, can be scanned straight from the file
by passing its path to [`scan_sequences()`][universalmotif.scan_sequences].

## Comparing motifs

Finally, [`compare_motifs()`][universalmotif.compare_motifs] measures how similar motifs are to one another
by the Pearson correlation of their aligned columns, giving a matrix by
default or a table of the best alignments with P-values:

In [ ]:
like_sites = ["TGAGTCA", "TGAGTCA", "TGACTCT"]
ap1_like = um.create_motif(like_sites, pseudocount=1, name="AP-1 like")
um.compare_motifs([ap1, ap1_like, creb]).round(3)

## Where next

This only covers the basics, and the other guides go further. [Sequence
motifs](motif-basics.ipynb) explains the motif types and how they relate to
one another, [Working with motifs](working-with-motifs.ipynb) and [Working
with sequences](sequences.ipynb) cover the everyday tools, and [Scanning and
enrichment](scanning-and-enrichment.ipynb), [Comparing motifs and
P-values](comparison-and-pvalues.ipynb) and [Choosing a
background](background-models.ipynb) go into the analyses. For whole
genomes, see [Scanning large genomes](large-genomes.ipynb), and [A ChIP-seq
workflow](chipseq-workflow.ipynb) puts most of this together on real peaks,
and [Building a motif library](building-a-library.ipynb) gathers motifs from
public databases.
The [API reference](../reference/index.md) describes every function.